# Carpark Data — Research

Building a parking app for Singapore (GoPark). The goal: a **single unified dataset** of every carpark in Singapore with locations, parking rates, and real-time availability.

The problem is there's no single source that has everything. Government data has locations and live availability but no rates. Community sites have rates but patchy coverage. Commercial sites have rates but use completely different ID systems. Each source fills gaps the others don't.

In [1]:
import os
import requests
import pandas as pd
import json
from dotenv import load_dotenv

load_dotenv()

True

## Government Data — data.gov.sg

The obvious starting point. Singapore's open data portal has a [carpark information dataset](https://data.gov.sg/datasets/d_23f946fa557947f93a8043bbef41dd09/view) with every HDB carpark — around 2000 entries. Coordinates, addresses, carpark type, parking system, free parking periods.

The API uses CKAN's `datastore_search` — you just pass the resource ID and a limit.

In [2]:
gov_url = "https://data.gov.sg/api/action/datastore_search"
params = {"resource_id": "d_23f946fa557947f93a8043bbef41dd09", "limit": 3}
r = requests.get(gov_url, params=params, timeout=30).json()
pd.DataFrame(r["result"]["records"]).T

,0,1,2
_id,1,2,3
car_park_no,ACB,ACM,AH1
address,BLK 270/271 ALBERT CENTRE BASEMENT CAR PARK,BLK 98A ALJUNIED CRESCENT,BLK 101 JALAN DUSUN
x_coord,30314.7936,33758.4143,29257.7203
y_coord,31490.4942,33695.5198,34500.3599
car_park_type,BASEMENT CAR PARK,MULTI-STOREY CAR PARK,SURFACE CAR PARK
type_of_parking_system,ELECTRONIC PARKING,ELECTRONIC PARKING,ELECTRONIC PARKING
short_term_parking,WHOLE DAY,WHOLE DAY,WHOLE DAY
free_parking,NO,SUN & PH FR 7AM-10.30PM,SUN & PH FR 7AM-10.30PM
night_parking,YES,YES,YES


The coordinates (`x_coord`, `y_coord`) aren't latitude/longitude — they're in **SVY21** (EPSG:3414), Singapore's local coordinate reference system. Everything else in our pipeline uses WGS84, so we need to convert with `pyproj`.

In [3]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True)

sample = r["result"]["records"][0]
x, y = float(sample["x_coord"]), float(sample["y_coord"])
lng, lat = transformer.transform(x, y)

print(f"SVY21:  x={x}, y={y}")
print(f"WGS84:  lat={lat:.6f}, lng={lng:.6f}")
print(f"Address: {sample['address']}")

SVY21:  x=30314.7936, y=31490.4942
WGS84:  lat=1.301063, lng=103.854118
Address: BLK 270/271 ALBERT CENTRE BASEMENT CAR PARK


Now fetch all of them and convert.

In [4]:
params_full = {"resource_id": "d_23f946fa557947f93a8043bbef41dd09", "limit": 3000}
gov_data = requests.get(gov_url, params=params_full, timeout=30).json()
gov_df = pd.DataFrame(gov_data["result"]["records"])
gov_df["x_coord"] = pd.to_numeric(gov_df["x_coord"], errors="coerce")
gov_df["y_coord"] = pd.to_numeric(gov_df["y_coord"], errors="coerce")
gov_df = gov_df.dropna(subset=["x_coord", "y_coord"])
gov_df = gov_df[(gov_df["x_coord"] > 0) & (gov_df["y_coord"] > 0)]

lngs, lats = transformer.transform(gov_df["x_coord"].values, gov_df["y_coord"].values)
gov_df["lat"] = lats
gov_df["lng"] = lngs

print(f"{len(gov_df)} government carparks loaded")
gov_df[["car_park_no", "address", "lat", "lng", "car_park_type", "type_of_parking_system", "free_parking"]].head(10)

2265 government carparks loaded


,car_park_no,address,lat,lng,car_park_type,type_of_parking_system,free_parking
0,ACB,BLK 270/271 ALBERT CENTRE BASEMENT CAR PARK,1.301063,103.854118,BASEMENT CAR PARK,ELECTRONIC PARKING,NO
1,ACM,BLK 98A ALJUNIED CRESCENT,1.321004,103.885061,MULTI-STOREY CAR PARK,ELECTRONIC PARKING,SUN & PH FR 7AM-10.30PM
2,AH1,BLK 101 JALAN DUSUN,1.328283,103.844620,SURFACE CAR PARK,ELECTRONIC PARKING,SUN & PH FR 7AM-10.30PM
3,AK19,BLOCK 253 ANG MO KIO STREET 21,1.369091,103.834985,SURFACE CAR PARK,COUPON PARKING,NO
4,AK31,BLK 302/348 ANG MO KIO STREET 31,1.366120,103.846636,SURFACE CAR PARK,COUPON PARKING,NO
5,AK52,BLK 513 ANG MO KIO STREET 53,1.372439,103.850296,SURFACE CAR PARK,COUPON PARKING,NO
6,AK9,ANG MO KIO AVENUE 9,1.383599,103.848368,SURFACE CAR PARK,COUPON PARKING,NO
7,AM14,BLK 227 ANG MO KIO STREET 23,1.368741,103.840301,MULTI-STOREY CAR PARK,ELECTRONIC PARKING,NO
8,AM16,BLK 256A ANG MO KIO STREET 21,1.370350,103.835718,MULTI-STOREY CAR PARK,ELECTRONIC PARKING,SUN & PH FR 7AM-10.30PM
9,AM18,BLK 308C ANG MO KIO AVENUE 1,1.365540,103.844619,MULTI-STOREY CAR PARK,ELECTRONIC PARKING,NO


### Real-time Availability

There's a separate endpoint for live lot availability. Returns current available lots and total capacity, broken down by lot type:

| Code | Type |
|------|------|
| **C** | Cars |
| **Y** | Motorcycles |
| **H** | Heavy Vehicles |
| **S** | Season Parking |

Both APIs use the same `car_park_no` identifiers (like `A0021`), so linking is trivial — just a dictionary lookup.

In [5]:
DATAGOVSG_KEY = os.environ.get("DATAGOVSG_KEY", "")

avail_url = "https://api.data.gov.sg/v1/transport/carpark-availability"
avail_headers = {"x-api-key": DATAGOVSG_KEY} if DATAGOVSG_KEY else {}
avail_data = requests.get(avail_url, headers=avail_headers, timeout=30).json()
carpark_data = avail_data["items"][0]["carpark_data"]

print(f"{len(carpark_data)} carparks with live availability")
for cp in carpark_data[:3]:
    print(f"  {cp['carpark_number']}: {cp['carpark_info']}")

2015 carparks with live availability
  HE12: [{'total_lots': '105', 'lot_type': 'C', 'lots_available': '101'}]
  HLM: [{'lot_type': 'C', 'total_lots': '583', 'lots_available': '264'}]
  RHM: [{'lot_type': 'C', 'total_lots': '329', 'lots_available': '153'}]


Linking them together by `car_park_no`:

In [6]:
sample_id = gov_df.iloc[0]["car_park_no"]
gov_info = gov_df[gov_df["car_park_no"] == sample_id].iloc[0]
avail_info = next((cp for cp in carpark_data if cp["carpark_number"] == sample_id), None)

print(f"Carpark {sample_id}")
print(f"  Address:  {gov_info['address']}")
print(f"  Location: {gov_info['lat']:.6f}, {gov_info['lng']:.6f}")
print(f"  Type:     {gov_info['car_park_type']}")
if avail_info:
    for lot in avail_info["carpark_info"]:
        print(f"  Lots ({lot['lot_type']}): {lot['lots_available']}/{lot['total_lots']} available")

Carpark ACB
  Address:  BLK 270/271 ALBERT CENTRE BASEMENT CAR PARK
  Location: 1.301063, 103.854118
  Type:     BASEMENT CAR PARK
  Lots (C): 23/91 available


## LTA DataMall — The Missing Carparks

The data.gov.sg API only covers **HDB** carparks. But Singapore has hundreds of commercial carparks in malls and private developments that aren't in that dataset.

LTA's [DataMall](https://datamall.lta.gov.sg/content/datamall/en.html) has a CarPark Availability API that covers **all three agencies**:

| Agency | Coverage | ID Format | Example |
|--------|----------|-----------|---------|
| **HDB** | Public housing carparks | Same as gov `car_park_no` | `KB7` |
| **LTA** | Major malls (Orchard, Marina, HarbourFront, Jurong Lake District) | Sequential integer | `1` |
| **URA** | Commercial developments | Alphanumeric | `A0007` |

The HDB records overlap with what we already have, but LTA and URA carparks are completely new — Suntec City, VivoCity, ION Orchard, etc. These are some of the most-used carparks in Singapore.

One catch: the API doesn't return total capacity, only available lots. And LTA/URA IDs can collide with HDB `car_park_no` format, so we prefix them (`LTA-1`, `URA-A0007`).

In [7]:
DATAMALL_API = os.environ["DATAMALL_API"]

datamall_url = "https://datamall2.mytransport.sg/ltaodataservice/CarParkAvailabilityv2"
headers = {"AccountKey": DATAMALL_API}

all_dm = []
skip = 0
while True:
    resp = requests.get(datamall_url, headers=headers, params={"$skip": skip}, timeout=30)
    batch = resp.json().get("value", [])
    if not batch:
        break
    all_dm.extend(batch)
    skip += 500

dm_df = pd.DataFrame(all_dm)
print(f"{len(dm_df)} records from DataMall")
print(f"\nAgency breakdown:")
print(dm_df["Agency"].value_counts().to_string())
print(f"\nLot types:")
print(dm_df["LotType"].value_counts().to_string())

2593 records from DataMall

Agency breakdown:
Agency
HDB    2401
URA     152
LTA      40

Lot types:
LotType
C    2112
Y     264
H     215
S       2


In [8]:
lta_carparks = dm_df[dm_df["Agency"] == "LTA"][["CarParkID", "Development", "Area", "Location", "LotType", "AvailableLots"]].drop_duplicates(subset=["CarParkID", "LotType"])
ura_carparks = dm_df[dm_df["Agency"] == "URA"][["CarParkID", "Development", "Location", "LotType", "AvailableLots"]].drop_duplicates(subset=["CarParkID", "LotType"])

print(f"LTA carparks (malls/commercial): {lta_carparks['CarParkID'].nunique()} unique")
print(f"URA carparks (developments):     {ura_carparks['CarParkID'].nunique()} unique")

print(f"\nLTA Areas:")
print(lta_carparks["Area"].value_counts().to_string())

print(f"\nSample LTA carparks:")
for _, row in lta_carparks.head(6).iterrows():
    print(f"  LTA-{row['CarParkID']:>3s}  {row['Development']:<40s} {row['Area']:<20s} [{row['LotType']}] {row['AvailableLots']} avail")

print(f"\nSample URA carparks:")
for _, row in ura_carparks.head(6).iterrows():
    print(f"  URA-{row['CarParkID']:<6s}  {row['Development']:<40s} [{row['LotType']}] {row['AvailableLots']} avail")

LTA carparks (malls/commercial): 40 unique
URA carparks (developments):     89 unique

LTA Areas:
Area
Orchard               19
Others                 8
Marina                 6
Harbfront              5
JurongLakeDistrict     2

Sample LTA carparks:
  LTA-  2  Marina Square                            Marina               [C] 983 avail
  LTA-  3  Raffles City                             Marina               [C] 366 avail
  LTA-  4  Esplanade                                Marina               [C] 667 avail
  LTA-  5  Millenia Singapore                       Marina               [C] 902 avail
  LTA-  6  Singapore Flyer                          Marina               [C] 250 avail
  LTA- 16  VivoCity P3                              Harbfront            [C] 267 avail

Sample URA carparks:
  URA-A0007   ANGULLIA PARK OFF STREET                 [Y] 0 avail
  URA-A0007   ANGULLIA PARK OFF STREET                 [C] 224 avail
  URA-A0011   ARMENIAN STREET OFF STREET               [C] 9 avail
  U

### ID Cross-Reference

HDB carpark IDs from DataMall should match the ones we already have from data.gov.sg. Let's verify and see how many are new.

In [9]:
hdb_dm = dm_df[dm_df["Agency"] == "HDB"]["CarParkID"].unique()
gov_ids = set(gov_df["car_park_no"].values)
dm_hdb_ids = set(hdb_dm)

overlap = gov_ids & dm_hdb_ids
only_gov = gov_ids - dm_hdb_ids
only_dm = dm_hdb_ids - gov_ids

print(f"HDB carpark ID overlap:")
print(f"  Both:          {len(overlap)}")
print(f"  Only gov:      {len(only_gov)}")
print(f"  Only DataMall: {len(only_dm)}")

if only_dm:
    print(f"\n  DataMall-only HDB IDs (sample): {sorted(only_dm)[:10]}")
if only_gov:
    print(f"  Gov-only IDs (sample):          {sorted(only_gov)[:10]}")

print(f"\nNew carparks from DataMall:")
print(f"  LTA: {lta_carparks['CarParkID'].nunique()} (commercial malls)")
print(f"  URA: {ura_carparks['CarParkID'].nunique()} (private developments)")
print(f"  Total new: {lta_carparks['CarParkID'].nunique() + ura_carparks['CarParkID'].nunique()}")

HDB carpark ID overlap:
  Both:          1984
  Only gov:      281
  Only DataMall: 0
  Gov-only IDs (sample):          ['A1', 'A23', 'A30A', 'A31A', 'A52L', 'A54', 'A56', 'A57', 'A58', 'A89']

New carparks from DataMall:
  LTA: 40 (commercial malls)
  URA: 89 (private developments)
  Total new: 129


## Parkaholic

Found [parkaholic.sg](https://parkaholic.sg) while searching for Singapore carpark APIs. It's a community-driven site that aggregates parking info across Singapore — turns out it has a lot more carpark entries than the government dataset (~4700 vs ~2000).

The best part: it uses the **same government `car_park_no` IDs**. So we can directly enrich our existing records without any matching logic.

Three useful endpoints:

| Endpoint | What it gives us |
|----------|------------------|
| `data.json` | All carpark locations with lot types |
| `/api/dump` | Live availability snapshot |
| `/api/history` | Rate history per carpark (POST) |

In [10]:
park_data = requests.get("https://parkaholic.sg/data.json", timeout=30).json()

print(f"{len(park_data)} parkaholic entries")
for cpid in list(park_data.keys())[:3]:
    print(f"\n  {cpid}:")
    print(f"    {json.dumps(park_data[cpid], indent=4)}")

4043 parkaholic entries

  1:
    {
    "Development": "Suntec City",
    "Location": "1.29375 103.85718",
    "LotType": [
        "Car",
        "Motorcycle"
    ]
}

  2:
    {
    "Development": "Marina Square",
    "Location": "1.29115 103.85728",
    "LotType": [
        "Car",
        "Motorcycle"
    ]
}

  3:
    {
    "Development": "Raffles City",
    "Location": "1.29382 103.85319",
    "LotType": [
        "Car",
        "Motorcycle"
    ]
}


In [11]:
dump_data = requests.get("https://parkaholic.sg/api/dump", timeout=30).json()
dump_entries = dump_data.get("resp", {})

print(f"{len(dump_entries)} entries in availability dump")
sample_key = list(dump_entries.keys())[0]
print(f"\nSample ({sample_key}):")
print(json.dumps(dump_entries[sample_key], indent=2))

2114 entries in availability dump

Sample (2):
{
  "CarParkID": "2",
  "Area": "Marina",
  "Development": "Marina Square",
  "Location": "1.29115 103.85728",
  "AvailableLots": 983,
  "LotType": "C",
  "Agency": "LTA",
  "_id": "6a18291a60af1fc643b97b09"
}


The rate history endpoint takes a POST with the carpark ID. It returns weekday rates, saturday rates, sunday/PH rates, and special remarks — all as free-text strings.

In [12]:
rate_resp = requests.post(
    "https://parkaholic.sg/api/history",
    json={"CarParkID": "A0021"},
    headers={"content-type": "application/json", "origin": "https://parkaholic.sg"},
    timeout=10,
).json()

print("Rate data for A0021:")
print(json.dumps(rate_resp.get("resp", {}), indent=2))

Rate data for A0021:
{
  "CarParkID": "A0021",
  "history": {
    "0": 16,
    "1": 16,
    "2": 16,
    "3": 16,
    "4": 16,
    "5": 16,
    "6": 16,
    "7": 16,
    "8": 13,
    "9": 0,
    "10": 0,
    "11": 0,
    "12": 0,
    "13": 0,
    "14": 0,
    "15": 0,
    "16": 0,
    "17": 0,
    "18": 2,
    "19": 0,
    "20": 0,
    "21": 11,
    "22": 15,
    "23": 16
  },
  "last_ping": "2026-05-28T10:39:05.749Z",
  "saturday_rate": "- [07.00 AM to 05.00 PM]: $1.20 per 30 mins\n- [05.00 PM to 10.30 PM]: $0.60 per 30 mins\n- [10.30 PM to 07.00 AM]: $0.60 per 30 mins\n- [10.30 PM to 07.00 AM]: $5.00 per 510 mins",
  "sunday_publicholiday_rate": "- [07.00 AM to 05.00 PM]: $0.60 per 30 mins\n- [05.00 PM to 10.30 PM]: $0.60 per 30 mins\n- [10.30 PM to 07.00 AM]: $0.60 per 30 mins\n- [10.30 PM to 07.00 AM]: $5.00 per 510 mins",
  "weekdays_rate_1": "- [07.00 AM to 05.00 PM]: $1.50 per 30 mins\n- [05.00 PM to 10.30 PM]: $0.60 per 30 mins\n- [10.30 PM to 07.00 AM]: $0.60 per 30 mins\n- [1

### The Malaysia Problem

Parkaholic includes carparks in Johor Bahru and other Malaysian areas near the border. Obviously these don't belong in a Singapore parking app.

I built a Singapore boundary polygon by merging the official data.gov.sg coastline with OpenStreetMap data. Any carpark outside this polygon gets excluded.

In [13]:
from shapely.geometry import shape, Point

with open("singapore.geojson") as f:
    sg_geojson = json.load(f)

border = shape(sg_geojson["features"][0]["geometry"])

test_sg = (1.3521, 103.8198)
test_jb = (1.4927, 103.7414)
print(f"Singapore CBD {test_sg}: inside = {border.contains(Point(test_sg[1], test_sg[0]))}")
print(f"Johor Bahru   {test_jb}: inside = {border.contains(Point(test_jb[1], test_jb[0]))}")

excluded = 0
for cpid, info in park_data.items():
    parts = info.get("Location", "").split()
    if len(parts) == 2:
        try:
            lat, lng = float(parts[0]), float(parts[1])
            if not border.contains(Point(lng, lat)):
                excluded += 1
        except ValueError:
            pass

print(f"\n{excluded} parkaholic entries outside Singapore (excluded)")
print(f"{len(park_data) - excluded} entries within Singapore")

Singapore CBD (1.3521, 103.8198): inside = True
Johor Bahru   (1.4927, 103.7414): inside = False



267 parkaholic entries outside Singapore (excluded)
3776 entries within Singapore


## SGCarMart

Found [sgcarmart.com/carpark](https://www.sgcarmart.com/carpark) — they have a carpark section with ~1100 carparks and detailed parking rate text for weekdays, evenings, weekends.

Their API is at `sgcarmart.com/api/carpark/fetch-carpark-detail-data?id={id}` but it's behind Cloudflare and rate-limits aggressively.

The catch: their IDs are just sequential integers (`1`, `2`, `3`...) with **no relation** to the government `car_park_no` system. This is where things get interesting.

In [14]:
sgc_df = pd.read_csv("rates/rates.csv", index_col=0)
print(f"{len(sgc_df)} SGCarMart carparks loaded from CSV")
sgc_df[["id", "name", "latitude", "longitude", "wd1", "sat"]].head(5)

1060 SGCarMart carparks loaded from CSV


,id,name,latitude,longitude,wd1,sat
0,1,National Library Building,1.297574,103.854034,"$1.20 for 1st 30min, $0.04/ subsequent min fro...",Charges same as weekdays
1,2,National Skin Centre,1.319699,103.849083,"$2.40 for 1st hr, $0.20 for next subsequent 5m...",Carpark closed
2,3,National University Hospital (NUH),1.293189,103.783386,$0.036/min from 12midnight to 12midnight the f...,Charges same as weekdays
3,4,Negara Hotel,1.307312,103.829918,"$6.00 for 1st hr, $3.00 for next subsequent 30...",Charges same as weekdays
4,5,Northpoint City,1.429909,103.835777,"$1.40 for 1st hr, $0.80 for next subsequent 30...","$1.50 for 1st hr, $0.80 for next subsequent 30..."
